# Base modelling

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, confusion_matrix,
    accuracy_score, f1_score, recall_score, precision_score,
    average_precision_score, brier_score_loss
)

# ../ navigates up from notebooks/ to the repo root
df = pd.read_parquet("../data/engineered/parquet_saves/me_merged_base.parquet")
print(df.columns.tolist())

In [ ]:
id_cols = ['bank_id', 'country_iso', 'year']
target = 'crisis'
macro_cols = ['NGDP_RPCH','NGDPD', 'PCPIPCH', 'LUR', 'BCA_NGDPD', 'BIS_REER', 'RXF11FX_REVS', 'reer_g', 'fx_g']
micro_cols = [c for c in df.columns if c not in id_cols + [target] + macro_cols]
all_cols = micro_cols + macro_cols

print("Micro:", micro_cols)
print("Macro:", macro_cols)
print(f"{len(micro_cols)} micro, {len(macro_cols)} macro, {len(all_cols)} integrated features")

# Collinearity identification

## VIF

In [ ]:
def compute_vif(df, cols):
    tmp = df[cols].dropna() #selects only columns you want to check, and drops na since it cannot handle missing values
    vif_data = pd.DataFrame({
        'feature': cols,
        'VIF': [variance_inflation_factor(tmp.values, i) for i in range(len(cols))] #takes full feature matrix and index of column to check, then returns VIF score
    })
    return vif_data.sort_values('VIF', ascending=False)

In [ ]:
print(df[macro_cols].isnull().sum())
print("Total rows:", len(df))

In [ ]:
print("=== Micro ===")
print(compute_vif(df, micro_cols))

print("\n=== Macro ===")
print(compute_vif(df, macro_cols))

print("\n=== Integrated ===")
print(compute_vif(df, all_cols))

## Belsley collinearity diagnostics

In [ ]:
def colldiag(X): 
    """
    Belsley (1980) collinearity diagnostics — port of R's VisCollin::colldiag().

    Scales each column of X to unit length (standard Belsley preprocessing),
    then computes condition indices and variance decomposition proportions.

    Parameters
    ----------
    X : pd.DataFrame — feature matrix, no target column, no NaNs

    Returns
    -------
    pd.DataFrame with columns [CI, intercept, feat1, feat2, ...]
    Rows are components ordered from CI=1 (no collinearity) to largest CI.
    Flag: CI > 30 AND two variables both have VDP > 0.5 in same row → collinear pair.
    """
    
    #Intercept column
    arr = np.column_stack([np.ones(len(X)), X.values.astype(float)])
    col_names = ['intercept'] + list(X.columns)

    #Scale columns to unit length
    col_norms = np.sqrt((arr**2).sum(axis=0)) #L2 norm of each column
    col_norms[col_norms == 0] = 1 #avoid division by zero for constant columns
    arr = arr / col_norms

    #SVD, singular values returned in descending order
    _, s, Vt = np.linalg.svd(arr, full_matrices=False)
    V = Vt.T

    #Condition indices: largest singular value divided by each singular value'
    ci = s[0] / s # large n means near-zero singular value = near-collinearity

    #Variance decomposition proportions: proportion of variance of each variable in each component'
    numerators = (V ** 2) / (s ** 2) # variance of each variable in each component, scaled by singular values
    phi = numerators / numerators.sum(axis=1, keepdims=True) # normalize to get proportions

    #Output
    vdp = pd.DataFrame(phi.T, columns=col_names)
    vdp.insert(0, 'CI', ci)

    return vdp.round(3)

In [ ]:
def find_collinear_pairs(X, ci_threshold=30, vdp_threshold=0.5):
    """
    Runs Belsley diagnostics and returns collinear groups automatically.
    For each flagged row (CI > ci_threshold), finds non-intercept variables
    with VDP > vdp_threshold. Two or more in the same row = collinear group.
    The intercept is excluded by convention — Belsley always includes it
    but it should not be interpreted as part of a variable pair.
    """
    diag = colldiag(X)
    flagged = diag[diag['CI'] > ci_threshold]

    pairs = []
    for _, row in flagged.iterrows():
        suspects = [
            col for col in diag.columns
            if col not in ['CI', 'intercept'] and row[col] > vdp_threshold
        ]
        if len(suspects) >= 2:
            pairs.append({'CI': round(row['CI'], 1), 'variables': suspects})

    return pairs

In [ ]:
result = colldiag(df[micro_cols].dropna())
print(result[result['CI'] > 30])

In [ ]:
result = colldiag(df[macro_cols].dropna())
print(result[result['CI'] > 30])

In [ ]:
print("=== Micro — collinear pairs ===")
micro_pairs = find_collinear_pairs(df[micro_cols].dropna())
for pair in micro_pairs:
    print(f"  CI={pair['CI']}: {pair['variables']}")
if not micro_pairs:
    print("  None found")

print("\n=== Macro — collinear pairs ===")
macro_pairs = find_collinear_pairs(df[macro_cols].dropna())
for pair in macro_pairs:
    print(f"  CI={pair['CI']}: {pair['variables']}")
if not macro_pairs:
    print("  None found")

Conclusion: collinear pair is x_loans_gross & x_loans_net, both at 0.998. gross loans carry more information (underprovision) -> more distorted by accounting choices. Drop x_loans_net

## Cols to drop based on automation

In [ ]:
# Drop decisions per feature set — determined by Belsley CI + VDP + economic reasoning.
# When running on a new dataset, call find_collinear_pairs() and record decisions here.
DROPS = {
    'micro': [
        # Accounting identities: exact linear dependence confirmed by CI > 1e10. can be dropped given this and derived ratios are unit free and more comparable than in their raw form
        'x_assets', 'x_equity', 'x_liab', 'x_liab_equity',
        # CI=211, VDP≈1.0 for both: keep x_loans_gross (less distorted pre-crisis by provisioning)
        'x_loans_net',
        # CI=34, VDP=0.946 vs 0.791: keep inc_op (absolute earnings level signal)
        'exp_total',
    ],
    'macro': [],  # fill after inspecting macro_pairs output above
}

In [ ]:
micro_cols_clean = [c for c in micro_cols if c not in DROPS['micro']]
macro_cols_clean = [c for c in macro_cols if c not in DROPS['macro']]
all_cols_clean   = [c for c in all_cols   if c not in DROPS['micro'] + DROPS['macro']]

print(f"Micro:      {len(micro_cols)} → {len(micro_cols_clean)} variables after drops")
print(f"Macro:      {len(macro_cols)} → {len(macro_cols_clean)} variables after drops")
print(f"Integrated: {len(all_cols)} → {len(all_cols_clean)} variables after drops")
print(f"\nMicro clean: {micro_cols_clean}")
print(f"Macro clean: {macro_cols_clean}")

Sanity check

In [ ]:
print("=== NaN counts in data===")
nan_counts = df[micro_cols_clean + macro_cols_clean].isnull().sum()
print(nan_counts[nan_counts > 0])
print("\nTotal rows:", len(df))

# Country ISO helper

In [ ]:
def get_country_col(df):
    # Different datasets use different column names — check both
    for col in ['country_iso', 'country_iso3']:
        if col in df.columns:
            return col
    # raise is OUTSIDE the loop — only fires if neither column is found
    raise ValueError(f"No country column found — expected 'country_iso' or 'country_iso3', got: {df.columns.tolist()}")

# Country dummies + feature set definitions

Country dummies are only fed into LR

In [ ]:
country_col = get_country_col(df)

# drop_first=True drops one country to avoid the dummy variable trap (last is implicit reference)
# astype(int) converts bool to 0/1 — required for SimpleImputer compatibility
country_dummies = pd.get_dummies(df[country_col], prefix='country', drop_first=True).astype(int)
country_dummy_cols = country_dummies.columns.tolist()

# Merge dummies into df so they are available when the expanding window splits df into train/test
# NOTE: re-run the splits cell after this one if you have already computed splits
df = pd.concat([df.reset_index(drop=True), country_dummies.reset_index(drop=True)], axis=1)

# Feature sets — same columns for RF and XGBoost (tree models capture country heterogeneity implicitly)
feature_sets = {
    'micro':      micro_cols_clean,
    'macro':      macro_cols_clean,
    'integrated': all_cols_clean,
}

# LR gets country dummies added to give country-specific intercepts (random intercept per country)
feature_sets_lr = {
    name: cols + country_dummy_cols
    for name, cols in feature_sets.items()
}

print(f"Country column: '{country_col}' — {len(country_dummy_cols)} dummies (reference country dropped)")
print("\nFeature set sizes:")
for name in feature_sets:
    print(f"  {name}: {len(feature_sets[name])} features | LR: {len(feature_sets_lr[name])}")

# H3 — Interaction terms (LR only, integrated condition)

H3 tests whether the predictive contribution of micro-level indicators depends on prevailing
macroeconomic conditions. For LR, this is operationalised by adding explicit microÃ—macro
interaction terms to the integrated feature set (tree models capture interactions implicitly).

Three interactions are included, selected on economic theory:

1. **Capital adequacy Ã— GDP growth** (`c_eqratio Ã— NGDP_RPCH`):
   Capital buffers matter more in recessions. Thin equity when GDP contracts = disproportionate
   vulnerability (Barrell et al., 2010; Demyanyk & Hasan, 2010; GonzÃ¡lez-Hermosillo, 1999).

2. **Leverage Ã— Unemployment** (`c_leverage Ã— LUR`):
   Rising unemployment drives loan defaults, hitting leveraged banks hardest. Macro timing
   (unemployment) Ã— bank-specific amplifier (leverage) = the trigger mechanism described in
   Citterio (2024) and GonzÃ¡lez-Hermosillo (1999).

3. **Loan concentration Ã— Inflation** (`r_loans_assets Ã— PCPIPCH`):
   Inflation erodes borrower repayment capacity (Du Plessis, 2022). Banks with high loan-to-asset
   ratios absorb the full impact of inflation-driven credit quality deterioration (Betz et al., 2012).

In [ ]:
# Compute interaction terms on the full df before any splitting.
# NaN propagates naturally: if either base variable is NaN, the product is NaN.
# The SimpleImputer inside the training loop handles these NaN values
# the same way it handles missing values in any other feature column.

interaction_cols = ['cap_x_gdp', 'lev_x_unemp', 'loans_x_infl']

df['cap_x_gdp']    = df['c_eqratio']     * df['NGDP_RPCH']  # capital adequacy Ã— GDP growth
df['lev_x_unemp']  = df['c_leverage']    * df['LUR']         # leverage Ã— unemployment
df['loans_x_infl'] = df['r_loans_assets'] * df['PCPIPCH']   # loan concentration Ã— inflation

# integrated_h3 = integrated feature set + 3 interaction terms + country dummies
# Only added to feature_sets_lr (LR only) — RF/XGBoost/MLP capture interactions implicitly
feature_sets_lr['integrated_h3'] = all_cols_clean + interaction_cols + country_dummy_cols

print(f"Interaction terms added: {interaction_cols}")
print(f"integrated_h3 feature set: {len(feature_sets_lr['integrated_h3'])} features "
      f"({len(all_cols_clean)} base + {len(interaction_cols)} interactions + {len(country_dummy_cols)} dummies)")
print(f"\nNaN counts in interaction columns:")
print(df[interaction_cols].isnull().sum())

# --- Piecewise / threshold interaction terms (addresses feedback #6) ---------
# The smooth products above force each interaction to scale LINEARLY with the
# macro value. A threshold interaction instead lets the micro effect STEP to a
# different value once the macro variable crosses a regime boundary -- capturing
# regime-switching interactions the smooth product cannot represent.
# Regime = macro variable on its "stress" side of the median. Direction per pair:
#   low growth = stress, high unemployment = stress, high inflation = stress.
# Threshold = whole-panel median (first-pass, mirrors the smooth terms above). If
# this form moves AUROC, promote the median to PER-FOLD before it enters the thesis.
thr_interaction_cols = ['cap_x_lowgrowth', 'lev_x_highunemp', 'loans_x_highinfl']

_gdp_med   = df['NGDP_RPCH'].median()
_unemp_med = df['LUR'].median()
_infl_med  = df['PCPIPCH'].median()

df['cap_x_lowgrowth']  = df['c_eqratio']      * (df['NGDP_RPCH'] < _gdp_med).astype(int)   # capital x low-growth regime
df['lev_x_highunemp']  = df['c_leverage']     * (df['LUR']       > _unemp_med).astype(int)  # leverage x high-unemployment regime
df['loans_x_highinfl'] = df['r_loans_assets'] * (df['PCPIPCH']   > _infl_med).astype(int)   # loans x high-inflation regime

feature_sets_lr['integrated_h3_thr'] = all_cols_clean + thr_interaction_cols + country_dummy_cols
print()
print(f"Threshold interaction terms added: {thr_interaction_cols}")
print(f"Regime medians -- GDP growth: {_gdp_med:.2f}, unemployment: {_unemp_med:.2f}, inflation: {_infl_med:.2f}")
print(f"integrated_h3_thr feature set: {len(feature_sets_lr['integrated_h3_thr'])} features")

# Expanding window split

In [ ]:
def expanding_window_split(df, min_train_years):
    years = sorted(df['year'].unique())
    splits = []

    for i, test_year in enumerate(years):
        if i < min_train_years:
            continue
        train = df[df['year'] < test_year]
        test = df[df['year'] == test_year]
        splits.append((train, test))
    return splits

In [ ]:
splits = expanding_window_split(df, min_train_years=5)
for train, test in splits[:3]:
    print(f"Train: {train['year'].min()}-{train['year'].max()}, Test: {test['year'].values[0]}")

# Results dataframe

In [ ]:
results = pd.DataFrame(columns = [
    'model', 'variant', 'dataset', 'test_year',
    'auroc', 'auprc', 'brier', 'best_threshold',
    'tp', 'fp', 'fn', 'tn',
    'f1', 'recall', 'precision',
    'miss_rate', 'false_alarm_rate', 'accuracy',
    'loss_07', 'loss_08', 'loss_09'
])

# Run_model helper

In [ ]:
def run_model(model, X_train, y_train, X_test, y_test, metadata, sample_weight=None, return_proba=False):
    # sample_weight is only used by MLP -- LR/RF/XGBoost handle imbalance via class_weight/scale_pos_weight
    fit_kwargs = {'sample_weight': sample_weight} if sample_weight is not None else {}
    model.fit(X_train, y_train, **fit_kwargs)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Find best threshold by maximising F1
    # np.where evaluates both branches eagerly, so the divide runs even where denom=0.
    # errstate suppresses the warning; np.where then correctly replaces those positions with 0.0.
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    denom     = precisions + recalls
    with np.errstate(invalid='ignore', divide='ignore'):
        f1_scores = np.where(denom > 0, 2 * precisions * recalls / denom, 0.0)
    best_threshold = thresholds[np.argmax(f1_scores[:-1])]
    y_pred = (y_proba >= best_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    auroc            = roc_auc_score(y_test, y_proba)
    auprc            = average_precision_score(y_test, y_proba)
    brier            = brier_score_loss(y_test, y_proba)
    f1               = f1_score(y_test, y_pred, zero_division=0)
    recall           = recall_score(y_test, y_pred, zero_division=0)
    precision        = precision_score(y_test, y_pred, zero_division=0)
    miss_rate        = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    accuracy         = accuracy_score(y_test, y_pred)

    loss_07 = 0.7 * miss_rate + 0.3 * false_alarm_rate
    loss_08 = 0.8 * miss_rate + 0.2 * false_alarm_rate
    loss_09 = 0.9 * miss_rate + 0.1 * false_alarm_rate

    result = {
        **metadata,
        'auroc':            auroc,
        'auprc':            auprc,
        'brier':            brier,
        'best_threshold':   best_threshold,
        'tp':               int(tp),
        'fp':               int(fp),
        'fn':               int(fn),
        'tn':               int(tn),
        'f1':               f1,
        'recall':           recall,
        'precision':        precision,
        'miss_rate':        miss_rate,
        'false_alarm_rate': false_alarm_rate,
        'accuracy':         accuracy,
        'loss_07':          loss_07,
        'loss_08':          loss_08,
        'loss_09':          loss_09,
    }

    # Only include raw probabilities when requested -- used by the main training loop
    # to populate proba_store for DeLong tests. Diagnostic loops (H3, no-dummies)
    # don't request them so their DataFrames stay clean.
    if return_proba:
        result['_y_proba'] = list(y_proba)
        result['_y_true']  = list(y_test)

    return result

# Training loop

In [ ]:
import optuna
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb

# Suppress per-trial Optuna output — only warnings and errors will print
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def get_objective(model_name, X_tr_raw, y_tr, X_vl_raw, y_vl, n_neg_ratio):
    """
    Returns an Optuna objective function for the given model type.

    Imputation (and scaling for LR/MLP) are done inside each trial, fitted on inner_train only.
    This mirrors the full pipeline and prevents any leakage from inner_val into inner_train.
    Objective: maximise AUROC on the inner validation set.

    Parameters
    ----------
    X_tr_raw, X_vl_raw : pd.DataFrame — raw (pre-imputed) feature matrices
    y_tr, y_vl         : pd.Series   — crisis labels
    n_neg_ratio        : float        — n_neg / n_pos in the full training window (for XGBoost/MLP)
    """
    def objective(trial):
        # Fit imputer on inner_train only — same leakage rule as the outer loop
        imputer = SimpleImputer(strategy='median')
        X_tr = imputer.fit_transform(X_tr_raw)
        X_vl = imputer.transform(X_vl_raw)

        if model_name == 'logistic_regression':
            C = trial.suggest_float('C', 1e-4, 10.0, log=True)
            # Scale inside trial — fit on inner_train only
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_vl = scaler.transform(X_vl)
            # L2 (Ridge): shrinks all coefficients toward zero but keeps all features.
            # Chosen over L1 because we want a consistent feature set across model classes
            # for cross-model SHAP comparison — L1 would zero out different features each window.
            # lbfgs is the recommended solver for L2; faster and more stable than saga for small-medium data.
            model = LogisticRegression(
                solver='lbfgs', C=C,
                max_iter=1000, class_weight='balanced', random_state=42
            )

        elif model_name == 'random_forest':
            model = RandomForestClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 500),
                max_depth=trial.suggest_int('max_depth', 3, 20),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
                max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
                class_weight='balanced', random_state=42,
            )

        elif model_name == 'xgboost':
            model = XGBClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 500),
                max_depth=trial.suggest_int('max_depth', 2, 10),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                subsample=trial.suggest_float('subsample', 0.5, 1.0),
                min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
                scale_pos_weight=n_neg_ratio,
                eval_metric='logloss', random_state=42
            )

        elif model_name == 'mlp':
            # MLP needs scaled inputs — gradient descent is sensitive to feature magnitude
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_vl = scaler.transform(X_vl)
            # suggest_categorical only accepts primitives — encode arch as string, decode to tuple
            arch = trial.suggest_categorical('hidden_layer_sizes', ['32', '64', '128', '64_32', '128_64'])
            # single-layer options (32/64/128): small to large capacity for limited panel data
            # two-layer funnels (64_32/128_64): standard progressive narrowing for tabular finance
            hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}[arch]
            model = MLPClassifier(
                hidden_layer_sizes=hidden_layer_sizes,
                learning_rate_init=trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
                alpha=trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
                max_iter=5000,
                tol=1e-3,  # relaxed from default 1e-4 — suppresses warnings on flat loss landscapes without affecting AUROC
                random_state=42
            )

        elif model_name == 'decision_tree':
            model = DecisionTreeClassifier(
            max_depth=trial.suggest_int('max_depth', 2, 15),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 30),
            class_weight='balanced', random_state=42
            )
    
        elif model_name == 'lightgbm':
            # Ensure consistent feature names for fit and predict
            # SimpleImputer may return a DataFrame in newer sklearn versions
            if not isinstance(X_tr, pd.DataFrame):
                X_tr = pd.DataFrame(X_tr, columns=X_tr_raw.columns)
                X_vl = pd.DataFrame(X_vl, columns=X_vl_raw.columns)
            model = lgb.LGBMClassifier(
            n_estimators=trial.suggest_int('n_estimators', 50, 500),
            max_depth=trial.suggest_int('max_depth', 2, 10),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int('num_leaves', 15, 63),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 30),
            scale_pos_weight=n_neg_ratio,
            random_state=42, verbose=-1
        )

        # MLP has no class_weight parameter — pass sample_weight to handle class imbalance
        # All other models already handle imbalance via class_weight / scale_pos_weight
        if model_name == 'mlp':
            sw = np.where(y_tr == 1, n_neg_ratio, 1.0)
            model.fit(X_tr, y_tr, sample_weight=sw)
        else:
            model.fit(X_tr, y_tr)
        y_proba = model.predict_proba(X_vl)[:, 1]
        return roc_auc_score(y_vl, y_proba)
        
    return objective

## What's not tuned (fixed by design)


Each model has parameters that are **held fixed** — they are design choices that apply across all windows and variants, not tuning targets.

| Model | Fixed parameter | Value | Why fixed |
|-------|----------------|-------|-----------|
| All | `random_state` | `42` | Reproducibility across runs |
| LR | `solver` | `lbfgs` | Best for L2 penalty; faster and more stable than `saga` on small-medium panel data |
| LR | `penalty` | `l2` | Chosen for SHAP cross-model comparability — L1 would zero out different features per window, making SHAP values incomparable |
| LR | `max_iter` | `1000` | Sufficient for convergence; avoids `ConvergenceWarning` |
| LR, RF, DT | `class_weight` | `'balanced'` | Imbalance correction — a modelling principle, not a tuning target. Sklearn computes inverse-frequency weights automatically |
| XGBoost, LightGBM | `scale_pos_weight` | `n_neg / n_pos` | Same rationale as `class_weight`; XGB/LGBM use this scalar instead. Computed per window so it tracks the actual class ratio in each training set |
| XGBoost | `eval_metric` | `'logloss'` | Standard for binary classification with `predict_proba` |
| MLP | `activation` | `'relu'` (default) | Near-universal default for tabular data; no theoretical reason to tune it here |
| MLP | `max_iter` | `5000` | Generous budget to allow convergence on small crisis datasets without inflating wall time |
| MLP | `tol` | `1e-3` | Relaxed from sklearn default (`1e-4`) — suppresses warnings on flat loss landscapes in short inner windows; does not affect AUROC |
| MLP | `sample_weight` | `n_neg / n_pos` | MLP has no `class_weight` param — sample weights replicate the same imbalance correction used by the other models |
| DT | `criterion` | `'gini'` (default) | Gini and entropy give near-identical splits in practice; not worth a tuning dimension |
| LightGBM | `verbose` | `-1` | Suppress per-tree console output only — no effect on training |

In [ ]:
MODEL_NAMES = ['logistic_regression', 'random_forest', 'xgboost', 'mlp', 'decision_tree', 'lightgbm']
N_TRIALS = 50  # Optuna trials per model per window

# Two variants:
#   baseline_t1— full training window, no exclusions
#   rt1        — exclude crisis year (t) and post-crisis year (t+1) from training, per country.
#                Non-crisis countries keep all their data even in crisis calendar years.
#                Rationale: crisis/post-crisis bank profiles are "disorderly adjustments"
#                not informative about pre-crisis vulnerability (Bussiere & Fratzscher 2006).
#                With duration labeling, RT1 targets the ONSET year + t+1 only (not the full
#                duration), because the sustained crisis period is itself informative.
VARIANTS = [('baseline_t1', False), ('rt1', True)]

all_results     = []
best_params_log = {}  # keyed by (variant, dataset, test_year, model)
proba_store     = {}  # (model, dataset, test_year, variant) → (y_true_list, y_proba_list)
fi_store        = {}  # (model, dataset, test_year, variant) → {feature: importance} (None for MLP)

for variant_name, use_rt1 in VARIANTS:
    for dataset_name, feat_cols in feature_sets.items():
        lr_cols = feature_sets_lr[dataset_name]
        print(f"\n=== Variant: {variant_name} | Dataset: {dataset_name} ===")

        for train_full, test in splits:
            test_year = test['year'].values[0]
            y_test    = test[target]

            if y_test.sum() == 0:
                print(f"  test={test_year} | SKIPPED — no crisis obs in test year")
                continue

            # RT1: build the set of (country, year) pairs to exclude — onset year + t+1.
            # We identify onset years as country-years where crisis first switches to 1
            # (i.e. crisis=1 but crisis=0 in the preceding year for that country).
            # Only those rows (and their t+1) are dropped; the rest of the crisis duration stays.
            if use_rt1:
                df_sorted   = train_full.sort_values([country_col, 'year'])
                prev_crisis = df_sorted.groupby(country_col)[target].shift(1).fillna(0)
                onset_mask  = (df_sorted[target] == 1) & (prev_crisis == 0)
                onset_rows  = df_sorted[onset_mask][[country_col, 'year']]
                exclude_set = (
                    set(zip(onset_rows[country_col], onset_rows['year'])) |
                    set(zip(onset_rows[country_col], onset_rows['year'] + 1))
                )
                train = train_full[~train_full.apply(
                    lambda r: (r[country_col], r['year']) in exclude_set, axis=1
                )]
                if train[target].sum() == 0:
                    print(f"  test={test_year} | RT1 SKIPPED — no crisis obs after onset filter")
                    continue
            else:
                train = train_full

            y_train     = train[target]
            n_pos       = (y_train == 1).sum()
            n_neg       = (y_train == 0).sum()
            n_neg_ratio = n_neg / n_pos if n_pos > 0 else 1.0

            # Inner split: last 2 years of the (possibly filtered) training window
            train_years     = sorted(train['year'].unique())
            inner_val_years = set(train_years[-2:])
            inner_train     = train[~train['year'].isin(inner_val_years)]
            inner_val       = train[train['year'].isin(inner_val_years)]
            y_inner_train   = inner_train[target]
            y_inner_val     = inner_val[target]

            can_tune = y_inner_val.sum() > 0 and y_inner_train.sum() > 0
            print(f"  test={test_year} | val={sorted(inner_val_years)} | crisis_in_val={can_tune}")

            for model_name in MODEL_NAMES:
                cols = lr_cols if model_name == 'logistic_regression' else feat_cols

                # --- Inner loop: Optuna tunes hyperparameters on inner_train/inner_val ---
                if can_tune:
                    objective = get_objective(
                        model_name,
                        inner_train[cols], y_inner_train,
                        inner_val[cols],   y_inner_val,
                        n_neg_ratio
                    )
                    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
                    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
                    best_params = study.best_params
                else:
                    best_params = {}

                best_params_log[(variant_name, dataset_name, test_year, model_name)] = best_params

                # --- Outer loop: refit on full (filtered) training window ---
                imputer     = SimpleImputer(strategy='median')
                X_train_imp = imputer.fit_transform(train[cols])
                X_test_imp  = imputer.transform(test[cols])

                if model_name == 'logistic_regression':
                    scaler        = StandardScaler()
                    X_train_final = scaler.fit_transform(X_train_imp)
                    X_test_final  = scaler.transform(X_test_imp)
                    model = LogisticRegression(
                        solver='lbfgs',
                        C=best_params.get('C', 1.0),
                        max_iter=1000, class_weight='balanced', random_state=42
                    )

                elif model_name == 'random_forest':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = RandomForestClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', None),
                        min_samples_leaf=best_params.get('min_samples_leaf', 1),
                        max_features=best_params.get('max_features', 'sqrt'),
                        class_weight='balanced', random_state=42
                    )

                elif model_name == 'xgboost':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = XGBClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', 6),
                        learning_rate=best_params.get('learning_rate', 0.1),
                        subsample=best_params.get('subsample', 1.0),
                        min_child_weight=best_params.get('min_child_weight', 1),
                        scale_pos_weight=n_neg_ratio,
                        eval_metric='logloss', random_state=42
                    )

                elif model_name == 'mlp':
                    scaler        = StandardScaler()
                    X_train_final = scaler.fit_transform(X_train_imp)
                    X_test_final  = scaler.transform(X_test_imp)
                    arch = best_params.get('hidden_layer_sizes', '128')
                    hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}.get(arch, (64,))
                    model = MLPClassifier(
                        hidden_layer_sizes=hidden_layer_sizes,
                        learning_rate_init=best_params.get('learning_rate_init', 1e-3),
                        alpha=best_params.get('alpha', 1e-4),
                        max_iter=5000,
                        tol=1e-3,  # relaxed from default 1e-4 — suppresses warnings on flat loss landscapes without affecting AUROC
                        random_state=42
                    )

                elif model_name == 'decision_tree':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = DecisionTreeClassifier(
                        max_depth=best_params.get('max_depth', 5),
                        min_samples_leaf=best_params.get('min_samples_leaf', 5),
                        class_weight='balanced', random_state=42
                    )

                elif model_name == 'lightgbm':
                    # Wrap as DataFrame: ensures consistent feature names for fit and predict
                    X_train_final = pd.DataFrame(X_train_imp, columns=cols)
                    X_test_final  = pd.DataFrame(X_test_imp,  columns=cols)
                    model = lgb.LGBMClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', 6),
                        learning_rate=best_params.get('learning_rate', 0.1),
                        num_leaves=best_params.get('num_leaves', 31),
                        subsample=best_params.get('subsample', 1.0),
                        min_child_samples=best_params.get('min_child_samples', 20),
                        scale_pos_weight=n_neg_ratio,
                        random_state=42, verbose=-1
                    )

                sw = np.where(y_train == 1, n_neg_ratio, 1.0) if model_name == 'mlp' else None
                result = run_model(
                    model, X_train_final, y_train, X_test_final, y_test,
                    metadata={
                        'model':     model_name,
                        'variant':   variant_name,
                        'dataset':   dataset_name,
                        'test_year': test_year,
                    },
                    sample_weight=sw,
                    return_proba=True,
                )
                # Store raw probabilities for DeLong test, then remove from result
                # so they don't appear as columns in the results DataFrame
                proba_store[(model_name, dataset_name, test_year, variant_name)] = (
                    result.pop('_y_true'), result.pop('_y_proba')
                )
                # Tree models expose feature_importances_, LR exposes coef_
                # MLP has no native importance (None here; use SHAP in NB13 if needed)
                if hasattr(model, 'feature_importances_'):
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = dict(zip(cols, model.feature_importances_))
                elif hasattr(model, 'coef_'):
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = dict(zip(cols, model.coef_[0]))
                else:
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = None
                all_results.append(result)

if not all_results:
    print("\nWARNING: no results collected — every test year had zero crisis observations.")
    print(f"Crisis counts by year:\n{df.groupby('year')[target].sum()}")
else:
    results = pd.DataFrame(all_results)
    print(f"\nCompleted: {len(results)} model-window evaluations")
    print(f"\nMean AUROC by model and dataset (baseline_t1):")
    print(
        results[results['variant'] == 'baseline_t1']
        .groupby(['model', 'dataset'])['auroc'].mean().round(3).unstack()
    )
    print(f"\nMean AUROC by model and dataset (rt1):")
    print(
        results[results['variant'] == 'rt1']
        .groupby(['model', 'dataset'])['auroc'].mean().round(3).unstack()
    )

# LR sanity check

# LR without country dummies — diagnostic

In [ ]:
# === Diagnostic: LR without country dummies ===
# Purpose: quantify how much of LR's high AUROC (~0.997) comes from country dummies
# encoding persistent crisis membership (16 countries in crisis 2008â€“2017) vs from
# the financial ratios themselves.
#
# Method: re-run the expanding window evaluation for LR only, using feature_sets
# (no dummies) instead of feature_sets_lr. Compare AUROC to the main results.
# If AUROC drops substantially, country dummies are driving performance, not ratios.

nodummy_results = []

for dataset_name, feat_cols in feature_sets.items():
    for train, test in splits:
        test_year = test['year'].values[0]
        y_train   = train[target]
        y_test    = test[target]

        if y_test.sum() == 0:
            continue

        n_pos       = (y_train == 1).sum()
        n_neg       = (y_train == 0).sum()
        n_neg_ratio = n_neg / n_pos

        # Tune C on inner validation window — same inner split as main loop
        train_years     = sorted(train['year'].unique())
        inner_val_years = set(train_years[-2:])
        inner_train     = train[~train['year'].isin(inner_val_years)]
        inner_val       = train[train['year'].isin(inner_val_years)]
        y_inner_train   = inner_train[target]
        y_inner_val     = inner_val[target]

        can_tune = y_inner_val.sum() > 0 and y_inner_train.sum() > 0

        if can_tune:
            objective = get_objective(
                'logistic_regression',
                inner_train[feat_cols], y_inner_train,
                inner_val[feat_cols],   y_inner_val,
                n_neg_ratio
            )
            study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
            best_C = study.best_params.get('C', 1.0)
        else:
            best_C = 1.0

        # Refit on full training window without dummies
        imputer     = SimpleImputer(strategy='median')
        X_train_imp = imputer.fit_transform(train[feat_cols])
        X_test_imp  = imputer.transform(test[feat_cols])
        scaler      = StandardScaler()
        X_train_sc  = scaler.fit_transform(X_train_imp)
        X_test_sc   = scaler.transform(X_test_imp)

        model = LogisticRegression(
            solver='lbfgs', C=best_C,
            max_iter=1000, class_weight='balanced', random_state=42
        )
        result = run_model(
            model, X_train_sc, y_train, X_test_sc, y_test,
            metadata={
                'model':     'lr_no_dummies',
                'variant':   'baseline_t1',
                'dataset':   dataset_name,
                'test_year': test_year,
            },
            return_proba=True,
        )
        # Store probabilities and coefficients so NB13 can compare LR with vs without dummies
        proba_store[('lr_no_dummies', dataset_name, test_year, 'baseline_t1')] = (
            result.pop('_y_true'), result.pop('_y_proba')
        )
        fi_store[('lr_no_dummies', dataset_name, test_year, 'baseline_t1')] = dict(zip(feat_cols, model.coef_[0]))
        nodummy_results.append(result)

nodummy_df = pd.DataFrame(nodummy_results)

# Compare mean AUROC: LR with dummies vs LR without dummies
lr_with = results[(results['model'] == 'logistic_regression') & (results['variant'] == 'baseline_t1')].groupby('dataset')['auroc'].mean()
lr_no   = nodummy_df.groupby('dataset')['auroc'].mean()

comparison = pd.DataFrame({
    'LR with dummies':    lr_with,
    'LR without dummies': lr_no,
}).round(3)
comparison['AUROC drop'] = (comparison['LR with dummies'] - comparison['LR without dummies']).round(3)

print("=== LR AUROC: with vs without country dummies ===")
print(comparison.to_string())
print()
print("Interpretation:")
print("  Large AUROC drop → country dummies are driving performance, not financial ratios")
print("  Small AUROC drop → financial ratios have genuine predictive power")

# Also show year-by-year for micro (most affected feature set)
print("\n=== Year-by-year AUROC for micro feature set ===")
with_yr  = results[(results['model'] == 'logistic_regression') & (results['dataset'] == 'micro') & (results['variant'] == 'baseline_t1')].set_index('test_year')['auroc']
no_yr    = nodummy_df[nodummy_df['dataset'] == 'micro'].set_index('test_year')['auroc']
print(pd.DataFrame({'with dummies': with_yr, 'without dummies': no_yr}).round(3).to_string())

In [ ]:
# === LR Sanity Check ===
# Two concerns flagged during initial run:
# (1) Suspiciously high AUROC (~0.997) for LR on micro — check whether this is year-specific
#     or consistent across the evaluation window (consistent = country dummies encoding crisis membership)
# (2) L2 regularisation shrinks all coefficients toward zero — verify that financial ratios
#     carry meaningful weight and the model is not relying almost entirely on country dummies

# (1) AUROC by year for LR
lr_results = results[(results['model'] == 'logistic_regression') & (results['variant'] == 'baseline_t1')]
print("=== LR AUROC by test_year and dataset (baseline_t1) ===")
print(lr_results.pivot_table(index='test_year', columns='dataset', values='auroc').round(3).to_string())

# (2) Refit LR on the last available split and inspect coefficients
last_train, last_test = splits[-1]
cols_inspect = feature_sets_lr['integrated']

imputer_inspect        = SimpleImputer(strategy='median')
X_train_imp_inspect    = imputer_inspect.fit_transform(last_train[cols_inspect])
scaler_inspect         = StandardScaler()
X_train_scaled_inspect = scaler_inspect.fit_transform(X_train_imp_inspect)

# best_params_log key is now (variant, dataset, test_year, model)
last_year  = last_test['year'].values[0]
last_best  = best_params_log.get(('baseline_t1', 'integrated', last_year, 'logistic_regression'), {})
C_inspect  = last_best.get('C', 1.0)

lr_inspect = LogisticRegression(
    solver='lbfgs', C=C_inspect,
    max_iter=1000, class_weight='balanced', random_state=42
)
lr_inspect.fit(X_train_scaled_inspect, last_train[target])

coef_df = pd.DataFrame({
    'feature': cols_inspect,
    'coef':    lr_inspect.coef_[0]
}).sort_values('coef', key=abs, ascending=False)

n_nonzero = (coef_df['coef'] != 0).sum()
print(f"\n=== LR coefficients — last window (test={last_year}), integrated, C={C_inspect:.4f} ===")
print(f"Non-zero: {n_nonzero} / {len(coef_df)} features")
print(coef_df.head(20).to_string(index=False))

# H3 analysis — integrated LR with vs without interaction terms

In [ ]:
# === H3 Analysis: LR integrated vs LR integrated + interaction terms ===
# Hypothesis H3: the predictive contribution of micro indicators depends on macro conditions.
# Test: does adding explicit microÃ—macro interaction terms to the integrated LR improve AUROC?
# If yes → interaction effects are present and the integrated model benefits from them.
# Coefficient signs on interaction terms are also interpreted (e.g. negative cap_x_gdp means
# the risk penalty for low capital is larger when GDP growth is positive — or vice versa).
import os

h3_results = []

for dataset_name in ['integrated', 'integrated_h3', 'integrated_h3_thr']:
    feat_cols_lr = feature_sets_lr[dataset_name]

    for train, test in splits:
        test_year = test['year'].values[0]
        y_train   = train[target]
        y_test    = test[target]

        if y_test.sum() == 0:
            continue

        n_pos       = (y_train == 1).sum()
        n_neg       = (y_train == 0).sum()

        train_years     = sorted(train['year'].unique())
        inner_val_years = set(train_years[-2:])
        inner_train     = train[~train['year'].isin(inner_val_years)]
        inner_val       = train[train['year'].isin(inner_val_years)]
        y_inner_train   = inner_train[target]
        y_inner_val     = inner_val[target]

        can_tune = y_inner_val.sum() > 0 and y_inner_train.sum() > 0

        if can_tune:
            objective = get_objective(
                'logistic_regression',
                inner_train[feat_cols_lr], y_inner_train,
                inner_val[feat_cols_lr],   y_inner_val,
                n_neg / n_pos
            )
            study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
            best_C = study.best_params.get('C', 1.0)
        else:
            best_C = 1.0

        imputer     = SimpleImputer(strategy='median')
        X_train_imp = imputer.fit_transform(train[feat_cols_lr])
        X_test_imp  = imputer.transform(test[feat_cols_lr])
        scaler      = StandardScaler()
        X_train_sc  = scaler.fit_transform(X_train_imp)
        X_test_sc   = scaler.transform(X_test_imp)

        model = LogisticRegression(
            solver='lbfgs', C=best_C,
            max_iter=1000, class_weight='balanced', random_state=42
        )
        result = run_model(
            model, X_train_sc, y_train, X_test_sc, y_test,
            metadata={
                'model':     'lr_h3',
                'dataset':   dataset_name,
                'test_year': test_year,
            }
        )
        h3_results.append(result)

h3_df = pd.DataFrame(h3_results)

# AUROC comparison: integrated vs integrated_h3
h3_auroc = h3_df.groupby('dataset')['auroc'].mean().round(3)
print("=== H3: LR AUROC — integrated vs integrated + interactions ===")
print(h3_auroc.to_string())
auroc_gain = h3_auroc.get('integrated_h3', 0) - h3_auroc.get('integrated', 0)
print(f"\nAUROC gain from interaction terms: {auroc_gain:+.3f}")
print("Positive → interaction terms improve discrimination (H3 supported)")
auroc_gain_thr = h3_auroc.get('integrated_h3_thr', 0) - h3_auroc.get('integrated', 0)
print(f"AUROC gain from THRESHOLD (piecewise) interaction terms: {auroc_gain_thr:+.3f}")
print("If the piecewise gain exceeds the smooth-product gain above, the interaction is")
print("regime-shaped -- addresses feedback #6. If both are ~0, the null is robust to form.")
print("Near zero → interactions do not add predictive value beyond main effects")

# Inspect interaction term coefficients on the last window
print(f"\n=== Interaction term coefficients — last window (test={last_year}) ===")
feat_h3 = feature_sets_lr['integrated_h3']
imp_h3  = SimpleImputer(strategy='median')
X_h3    = imp_h3.fit_transform(last_train[feat_h3])
sc_h3   = StandardScaler()
X_h3    = sc_h3.fit_transform(X_h3)

lr_h3 = LogisticRegression(
    solver='lbfgs', C=1.0,
    max_iter=1000, class_weight='balanced', random_state=42
)
lr_h3.fit(X_h3, last_train[target])

coef_h3 = pd.DataFrame({
    'feature': feat_h3,
    'coef':    lr_h3.coef_[0]
})

# Print only the interaction terms and top 5 base features for context
int_coefs = coef_h3[coef_h3['feature'].isin(interaction_cols)].copy()
top_coefs = coef_h3[~coef_h3['feature'].isin(interaction_cols)].reindex(
    coef_h3[~coef_h3['feature'].isin(interaction_cols)]['coef'].abs().sort_values(ascending=False).index
).head(5)

print("\nInteraction term coefficients:")
print(int_coefs.sort_values('coef', key=abs, ascending=False).to_string(index=False))
print("\nTop 5 base feature coefficients (for context):")
print(top_coefs.to_string(index=False))

# --- Threshold interaction coefficients — last window (for #6 discussion) ---
feat_h3_thr = feature_sets_lr['integrated_h3_thr']
imp_ht = SimpleImputer(strategy='median')
X_ht   = imp_ht.fit_transform(last_train[feat_h3_thr])
X_ht   = StandardScaler().fit_transform(X_ht)
lr_ht  = LogisticRegression(solver='lbfgs', C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
lr_ht.fit(X_ht, last_train[target])
coef_ht = pd.DataFrame({'feature': feat_h3_thr, 'coef': lr_ht.coef_[0]})
int_coefs_thr = coef_ht[coef_ht['feature'].isin(thr_interaction_cols)].copy()
print()
print("Threshold interaction term coefficients:")
print(int_coefs_thr.sort_values('coef', key=abs, ascending=False).to_string(index=False))

#Auto-save results after every run so they're never lost
os.makedirs('results_11a', exist_ok=True)
# Include lr_no_dummies in the main results parquet — single source of truth for NB13
results = pd.concat([results, nodummy_df], ignore_index=True)
results.to_parquet(f"results_11a/results_n{N_TRIALS}_trials_final.parquet")
print(f"[SAVED] results_11a/results_n{N_TRIALS}_trials_final.parquet")

## DeLong Test

In [ ]:
from scipy import stats as scipy_stats

def delong_test(y_true, y_score_a, y_score_b):
    """
    DeLong et al. (1988) non-parametric test for comparing two AUROCs.
    Returns (z, p): positive z means AUROC_a > AUROC_b; p is two-sided.
    Returns (nan, nan) when fewer than 2 positives or 2 negatives exist
    (covariance cannot be estimated with a single observation).

    Method: placement values (structural components) measure how well each
    positive score outranks each negative score. Their covariance across the
    two score vectors gives the variance of the AUROC difference — no parametric
    assumptions about score distributions needed.
    """
    y_true    = np.asarray(y_true,    dtype=int)
    y_score_a = np.asarray(y_score_a, dtype=float)
    y_score_b = np.asarray(y_score_b, dtype=float)

    pos = y_true == 1
    neg = y_true == 0
    m   = pos.sum()  # number of positives
    n   = neg.sum()  # number of negatives

    if m < 2 or n < 2:
        return np.nan, np.nan

    def placements(scores):
        s_pos = scores[pos]
        s_neg = scores[neg]
        # V10[i]: fraction of negatives outranked by positive i (ties split 50/50)
        V10 = np.array([(p > s_neg).mean() + 0.5 * (p == s_neg).mean() for p in s_pos])
        # V01[j]: fraction of positives that outrank negative j (ties split 50/50)
        V01 = np.array([(s_pos > nj).mean() + 0.5 * (s_pos == nj).mean() for nj in s_neg])
        return V10, V01

    V10_a, V01_a = placements(y_score_a)
    V10_b, V01_b = placements(y_score_b)

    auc_a = V10_a.mean()
    auc_b = V10_b.mean()

    # 2Ã—2 covariance matrices of the placement value pairs
    S10 = np.cov(np.vstack([V10_a, V10_b]), ddof=1)
    S01 = np.cov(np.vstack([V01_a, V01_b]), ddof=1)
    S   = S10 / m + S01 / n  # covariance matrix of (AUC_a, AUC_b)

    # Var(AUC_a - AUC_b) = S[0,0] + S[1,1] - 2*S[0,1]
    var_diff = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var_diff <= 0:
        return np.nan, np.nan

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * scipy_stats.norm.sf(abs(z))
    return float(z), float(p)

In [ ]:
# === DeLong analysis ===
# Primary use: test H1 — does the integrated model significantly outperform micro-only
# and macro-only? Positive z = integrated AUROC > comparison AUROC.
# Secondary use: RT1 robustness — does excluding onset years change performance significantly?
# p < 0.05 in the RT1 vs baseline comparison would mean the onset rows were inflating results.

delong_rows = []
test_years  = sorted(results['test_year'].unique())

for variant_name in ['baseline_t1', 'rt1']:
    for model_name in MODEL_NAMES:
        for test_year in test_years:

            # H1: integrated vs micro and integrated vs macro
            for label, cmp_dataset in [('integrated_vs_micro', 'micro'),
                                        ('integrated_vs_macro', 'macro')]:
                key_int = (model_name, 'integrated',  test_year, variant_name)
                key_cmp = (model_name, cmp_dataset,   test_year, variant_name)
                if key_int not in proba_store or key_cmp not in proba_store:
                    continue
                y_true, y_proba_int = proba_store[key_int]
                _,      y_proba_cmp = proba_store[key_cmp]
                z, p = delong_test(y_true, y_proba_int, y_proba_cmp)
                delong_rows.append({
                    'comparison': label,
                    'variant':    variant_name,
                    'model':      model_name,
                    'test_year':  test_year,
                    'z':          z,
                    'p':          p,
                    'sig_05':     bool(p < 0.05) if not np.isnan(p) else False,
                })

        # RT1 robustness: for each dataset, does baseline vs rt1 differ significantly?
        for dataset_name in feature_sets:
            for test_year in test_years:
                key_base = (model_name, dataset_name, test_year, 'baseline_t1')
                key_rt1  = (model_name, dataset_name, test_year, 'rt1')
                if key_base not in proba_store or key_rt1 not in proba_store:
                    continue
                y_true, y_proba_base = proba_store[key_base]
                _,      y_proba_rt1  = proba_store[key_rt1]
                z, p = delong_test(y_true, y_proba_base, y_proba_rt1)
                delong_rows.append({
                    'comparison': f'baseline_vs_rt1_{dataset_name}',
                    'variant':    'rt1_check',
                    'model':      model_name,
                    'test_year':  test_year,
                    'z':          z,
                    'p':          p,
                    'sig_05':     bool(p < 0.05) if not np.isnan(p) else False,
                })

delong_df = pd.DataFrame(delong_rows)

# Summary: mean z and % significant windows per (comparison, variant, model)
print("=== DeLong summary: mean z-statistic and % significant test windows ===")
summary = (
    delong_df
    .groupby(['comparison', 'variant', 'model'])
    .agg(mean_z=('z', 'mean'), pct_sig=('sig_05', 'mean'), n=('z', 'count'))
    .round(3)
)
print(summary.to_string())

# Year-by-year p-values for the main H1 comparison (integrated vs micro, baseline)
print("\n=== H1: p-values by test year — integrated vs micro (baseline_t1) ===")
h1_mic = delong_df[
    (delong_df['comparison'] == 'integrated_vs_micro') &
    (delong_df['variant']    == 'baseline_t1')
].pivot_table(index='test_year', columns='model', values='p').round(3)
print(h1_mic.to_string())
print("(p < 0.05 = integrated significantly outperforms micro-only in that year)")

# RT1 robustness summary: significant change means onset rows were inflating baseline results
print("\n=== RT1 robustness: % of windows where baseline vs rt1 differ significantly ===")
rt1_check = delong_df[delong_df['variant'] == 'rt1_check']
print(rt1_check.groupby(['model', 'comparison'])['sig_05'].mean().round(3).to_string())
print("(high % = onset-year exclusion materially changes predictions → baseline results were inflated)")

## Save all results

In [ ]:
import os
import pandas as pd

out_dir = "../outputs/results/11a"
os.makedirs(out_dir, exist_ok=True)

with pd.ExcelWriter(f"{out_dir}/results_11a_final.xlsx", engine="openpyxl") as writer:

    # --- Main results ---
    results.to_excel(writer, sheet_name="all_results", index=False)

    results[results["variant"] == "baseline_t1"].pivot_table(
        index="model", columns="dataset", values="auroc", aggfunc="mean"
    ).round(3).to_excel(writer, sheet_name="auroc_pivot_baseline")

    results[results["variant"] == "baseline_t1"].groupby(["model", "dataset"])[
        ["auroc", "f1", "recall", "precision", "miss_rate", "false_alarm_rate",
  "loss_08"]
    ].mean().round(3).to_excel(writer, sheet_name="mean_metrics_baseline")

    results.groupby(["variant", "model", "dataset"])["auroc"].mean().round(3)\
        .unstack("dataset").to_excel(writer, sheet_name="baseline_vs_rt1")

    # --- LR diagnostics ---
    comparison.to_excel(writer, sheet_name="lr_dummy_comparison")

    nodummy_df.to_excel(writer, sheet_name="lr_no_dummies_raw", index=False)

    coef_df.to_excel(writer, sheet_name="lr_coefficients", index=False)

    # --- H3 analysis ---
    h3_df.to_excel(writer, sheet_name="h3_raw", index=False)

    h3_auroc.to_frame("auroc").to_excel(writer, sheet_name="h3_auroc_summary")

    int_coefs.to_excel(writer, sheet_name="h3_interaction_coefs", index=False)

    int_coefs_thr.to_excel(writer, sheet_name="h3_interaction_coefs_thr", index=False)

    # --- DeLong ---
    delong_df.to_excel(writer, sheet_name="delong_raw", index=False)

    summary.to_excel(writer, sheet_name="delong_summary")

    h1_mic.to_excel(writer, sheet_name="delong_h1_pvalues")

print(f"Saved to {out_dir}/results_11a_final.xlsx")

# Save proba_store — raw (y_true, y_proba) per model/window; needed for PR/ROC curves in NB13
with open(f"{out_dir}/proba_store_final.pkl", "wb") as f:
    pickle.dump(proba_store, f)
print(f"[SAVED] {out_dir}/proba_store_final.pkl")

# Save best_params_log — hyperparameter choices per model/window; for methodology table in NB13
with open(f"{out_dir}/best_params_log_final.pkl", "wb") as f:
    pickle.dump(best_params_log, f)
print(f"[SAVED] {out_dir}/best_params_log_final.pkl")

# Convert fi_store to long-format DataFrame and save; needed for feature importance plots in NB13
fi_rows = []
for (mdl, ds, yr, var), fi in fi_store.items():
    if fi is None:
        continue
    for feat, imp in fi.items():
        fi_rows.append({"model": mdl, "dataset": ds, "test_year": yr,
                        "variant": var, "feature": feat, "importance": imp})
fi_df = pd.DataFrame(fi_rows)
fi_df.to_parquet(f"{out_dir}/fi_store_final.parquet")
print(f"[SAVED] {out_dir}/fi_store_final.parquet")